In [1]:
with open('data/spl_checkouts_by_title_2022-Q1.csv', encoding="utf-8", errors="replace") as file: # review the input file before this is ran
    data_list = file.readlines()

In [2]:
print(len(data_list))

88283


In [3]:
print(data_list[0:5])

['CheckoutYear,CheckoutMonth,Checkouts,Title,PublicationYear\n', '2022,1,3,<Unknown Title>,\n', '2022,1,2,Fast fix favorites.,[2021]\n', '2022,1,3,Witchery : embrace the witch within / Juliet Diaz.,[2019]\n', '2022,1,1,Everything changes : help for families of newly recovering addicts / Beverly Conyers.,2009.\n']


In [4]:
from collections import namedtuple # Enable use of namedtuple

records = [] # output list definition

for record in data_list[1:]: # loop to convert records in the data_list file. Remove the header item only.
    
    stats = (record.strip()).split(sep=',',maxsplit=3) # Split the full record string in 4 parts (stats + rest). Also remove the '\n' from all lines
    title_and_year = (stats.pop(3)).rsplit(sep=',',maxsplit=1) # Separate the right side of the string containing the book title and year
    num_stats = [int(number) for number in stats] # transform the stats into integers
    record_as_tuple = tuple(num_stats + title_and_year) # create tuples from the combined lists of stats, title and year
    records.append(record_as_tuple) # append all tuples to the output list

Library_records = namedtuple('Library_record', ['CheckoutYear','CheckoutMonth','Checkouts','Title','PublicationYear']) # define the structure of the named tuple
checkout_records = [Library_records(*item) for item in records] # finalize the creation of the checkout_records variable by transforming all tuples into named tuples
print(len(checkout_records)) # confirm that all records minus the header were included, expected 88282

88282


In [5]:
for record in checkout_records[0:10]:
    print(record)

Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=3, Title='<Unknown Title>', PublicationYear='')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=2, Title='Fast fix favorites.', PublicationYear='[2021]')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=3, Title='Witchery : embrace the witch within / Juliet Diaz.', PublicationYear='[2019]')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=1, Title='Everything changes : help for families of newly recovering addicts / Beverly Conyers.', PublicationYear='2009.')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=3, Title='"One-Punch Man. 14 / story by ONE ; art by Yusuke Murata ; translation, John Werry."', PublicationYear='[2018]')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=1, Title='"Sensing God : experiencing the divine in nature, food, music & beauty / Joel Clarkson."', PublicationYear='[2021]')
Library_record(CheckoutYear=2022, CheckoutMonth=1, Checkouts=1, Ti

In [6]:
# This query can be used to test functions
example_query = "cowardly lion AND vicious tiger OR bear NOT polar bear"

In [7]:
# The parse_search_query function will take a query string provided by the user. The function will parse
# the query by tokenizing and then return a list of tuples that contains the operator and the search term that
# it is related with

def parse_search_query(search_query):

# TOKEN RECOGNITION
    token_list = [] # create empty token list variable
    current_phrase = '' # create current phrase variable and assign empty string
    query_split = search_query.split() # create a list of all items in the search query to iterate with
    
    for word in query_split: 
        
        if word.upper() != 'AND' and word.upper() != 'OR' and word.upper() != 'NOT': # look for search phrases, not operators
            if current_phrase == '':
                current_phrase = word.lower() # first word has no preceding spaces  
            else:
                current_phrase = current_phrase + " " + word.lower() # add spaces between words
        
        else: # if an operator is found, build the token list appending the current phrase and operator
            token_list.append(current_phrase.lower())
            token_list.append(word.upper())
            current_phrase = '' # clear the current phrase to build a new one
    
    token_list.append(current_phrase.lower()) # when no more operators are found, append the final phrase

#TUPLES CREATION
    tuple_list = [] # create empty tuple list variable
    
    for item in range(0,(len(token_list)),2): # iterate in steps of 2 the full range of the token list
        
        if item == 0: # the first item is paired with None
            token_pair = (None,token_list[item])
            tuple_list.append(token_pair)
        
        else: # all other items are paired with the operator which is the preceding item
            token_pair = (token_list[item-1],token_list[item])
            tuple_list.append(token_pair)

    return tuple_list # the tuple list is returned as output of this function

parse_search_query(example_query)

[(None, 'cowardly lion'),
 ('AND', 'vicious tiger'),
 ('OR', 'bear'),
 ('NOT', 'polar bear')]

In [8]:
# These tuples can be used to test functions
testing_catalog = [
    (2021, 1, 1, "1 a lion roars", "[2021]"),
    (2022, 2, 2, "2 a tiger roars", "[2022]"),
    (2023, 3, 3, "3 a bear roars", "[2023]"),
    (2024, 4, 4, "4 yellow lions and orange tiger", "[2024]"),
    (2025, 5, 5, "5 brown bears and white tigers", "[2025]"),
    (2026, 6, 6, "6 angry bear, cowardly lion", "[2026]"),
    (2027, 7, 7, "7 ecology of lions, tigers, bears, and ohmys", "[2027]"),
]

In [9]:
# The has_title_in_list function receives the book title and the list of
# checkout record tuples as arguments and returns a boolean indicating if there
# is a record with that exact title in the list

def has_title_in_list(book_title, record_tuples):
    
    # loops through each tuple of the record_tuples argument for the title and adds to the list the matches with book_title
    title_matches = [record[3] for record in record_tuples if record[3].lower() == book_title.lower()]
    
    if title_matches == []: # provide the boolean value to return depending on the contents of the list
        matches = False # False if empty
    else:
        matches = True # True if not empty

    return matches # return the boolean

has_title_in_list('Witchery : embrace the witch within / Juliet Diaz.', checkout_records)

True

In [10]:
# The get_search_results function receives a search query that provides the search
# criteria and a list of tuples with the records to review and returns a list of 
# the tuples that match the search criteria

def get_search_results(search_query,record_query_tuples):
    
    parsed_query = parse_search_query(search_query) # Obtain the tuples for the search query entered

    results_so_far = [] # define a starting variable to capture interim results
    final_results = [] # define a starting variable to capture the final results

    for query_tuple in parsed_query: # loop through the query tuples
        
        if query_tuple[0] == None: # None records retrieved from the full set of records
            results_so_far = [record for record in record_query_tuples if query_tuple[1].lower() in record[3].lower()]
        elif query_tuple[0] == 'AND': # review records from current results that also match the new query
            and_items_to_keep = [record for record in results_so_far if query_tuple[1].lower() in record[3].lower()]
            results_so_far = [item for item in and_items_to_keep]
        elif query_tuple[0] == 'NOT': # review records from current results that do not math the new query
            not_items_to_keep = [record for record in results_so_far if query_tuple[1].lower() not in record[3].lower()]
            results_so_far = [item for item in not_items_to_keep]
        elif query_tuple[0] == 'OR': # 
            if final_results != []: # if the final results are empty, the has_title_in_list function doesn't work
                final_results = final_results + [result for result in results_so_far if not has_title_in_list(result[3], final_results)]
            else:
                final_results = results_so_far
            results_so_far = [] # reset the results so far after they're stored in the final results
            # Loop through the full set of records with the new query tied to "OR"
            results_so_far = [record for record in record_query_tuples if query_tuple[1].lower() in record[3].lower()]
    
    # Final addition to the final results
    if final_results != []: 
        final_results = final_results + [result for result in results_so_far if not has_title_in_list(result[3], final_results)]
    else:
        final_results = results_so_far 
    
    return final_results 

get_search_results(example_query, testing_catalog)

[(2023, 3, 3, '3 a bear roars', '[2023]'),
 (2025, 5, 5, '5 brown bears and white tigers', '[2025]'),
 (2026, 6, 6, '6 angry bear, cowardly lion', '[2026]'),
 (2027, 7, 7, '7 ecology of lions, tigers, bears, and ohmys', '[2027]')]

In [11]:
# This code can be used to check the accuracy of the get_search_results function. 
# Expected outcome is "True" for all test cases

correctness_check_list = [
    (2022, 1, 1, "lions", "[2022]"),
    (2022, 1, 1, "tigers", "[2022]"),
    (2022, 1, 1, "bears", "[2022]"),
    (2022, 1, 1, "lions tigers", "[2022]"),
    (2022, 1, 1, "tigers bears", "[2022]"),
    (2022, 1, 1, "lions bears", "[2022]"),
    (2022, 1, 1, "lions tigers bears", "[2022]"),
]

print(len(get_search_results("lions AND tigers", correctness_check_list)) == 2)
print(len(get_search_results("lions OR tigers", correctness_check_list)) == 6)
print(len(get_search_results("lions NOT tigers", correctness_check_list)) == 2)

print(len(get_search_results("lions AND tigers NOT bears", correctness_check_list)) == 1)
print(len(get_search_results("lions AND tigers OR bears", correctness_check_list)) == 5)
print(len(get_search_results("lions AND tigers AND bears", correctness_check_list)) == 1)

print(len(get_search_results("lions NOT tigers AND bears", correctness_check_list)) == 1)
print(len(get_search_results("lions NOT tigers OR bears", correctness_check_list)) == 5)
print(len(get_search_results("lions NOT tigers NOT bears", correctness_check_list)) == 1)

print(len(get_search_results("lions OR tigers AND bears", correctness_check_list)) == 5)
print(len(get_search_results("lions OR tigers NOT bears", correctness_check_list)) == 5)
print(len(get_search_results("lions OR tigers OR bears", correctness_check_list)) == 7)

True
True
True
True
True
True
True
True
True
True
True
True


In [12]:
# The search_catalog function uses the get_search_results function to obtain a list of tuples with a
# custom search result query from the user and presents the results formatted back to the user

# import the clear_output function to allow clearing old search results
from IPython.display import clear_output 

def search_catalog():

    continue_searching = True # initialization of looping variable to continue searching
        
    while continue_searching: # loop continuation start
        
        # Introduction message and instructions to user
        print("""\n\033[1;34mWelcome to the Q1 2022 Seattle Public Library Collection Catalog Search\033[0m\n\033[1mEnter your search criteria (titles or keywords) you're looking for in the catalog.\nYou can use the operators "AND", "OR" and "NOT" to combine search terms\033[0m""")
        
        search_query = input("Your query: ") # User input 

        search_results = get_search_results(search_query, checkout_records) # Obtain search results

        print("\n Your search criteria: \033[1m" + search_query + "\033[0m\n   returned a total of \033[1m" + str(len(search_results)) + "\033[0m records.\n")
        print("Here are the records found:\n")
        
        iteration = 1 # initialize a counting variable to differentiate output records
        
        for result in search_results: # start results formatting using the fields and values from the named tuples
            fields = result._fields
            print(result.__class__.__name__ + " " + str(iteration)) # Print record name and sequence
            print(fields[3],":",result.Title)
            print(fields[4],":",result.PublicationYear)
            print(fields[0],":",result.CheckoutYear,fields[1],":",result.CheckoutMonth,fields[2],":",result.Checkouts,"\n")
            iteration = iteration + 1 # increase by 1 the sequence iteration
    
        print("\033[1m\nWould you like to search again?\033[0m")
        again_choice = input("\nType Yes to continue, anything else to finish: ").lower()
        print(again_choice)
        if again_choice.startswith("y") and again_choice != '':
            print("\033[1m\nWould you like to keep your current search results?\033[0m")
            clear_choice = input("\nType Yes to keep them, anything else to remove them: ").lower()
            print(clear_choice)
            if clear_choice.startswith("y") and clear_choice != '':
                continue_searching = True
            else:
                continue_searching = True
                clear_output(wait=True) # clear search results
            search_query = []
            search_results = []
        else:
            continue_searching = False

In [13]:
search_catalog()


Welcome to the Q1 2022 Seattle Public Library Collection Catalog Search
Enter your search criteria (titles or keywords) you're looking for in the catalog.
You can use the operators "AND", "OR" and "NOT" to combine search terms


Your query:  harry potter



 Your search criteria: harry potter
   returned a total of 38 records.

Here are the records found:

Library_record 1
Title : Hari Potta to fujicho no kishidan = Harry Potter and the Order of the Phoenix / J.K. Rowling cho ; Matsuoka Yūko yaku.
PublicationYear : 2004.
CheckoutYear : 2022 CheckoutMonth : 1 Checkouts : 1 

Library_record 2
Title : The science of Harry Potter : how magic really works / Roger Highfield.
PublicationYear : 2002.
CheckoutYear : 2022 CheckoutMonth : 1 Checkouts : 1 

Library_record 3
Title : Harry Potter : a history of magic.
PublicationYear : 2018.
CheckoutYear : 2022 CheckoutMonth : 1 Checkouts : 7 

Library_record 4
Title : Harry Potter : the creature vault : the creatures and plants of the Harry Potter films / by Jody Revenson.
PublicationYear : 2014.
CheckoutYear : 2022 CheckoutMonth : 1 Checkouts : 3 

Library_record 5
Title : Knitting magic : more patterns from Hogwarts & beyond : an official Harry Potter knitting pattern book / Tanis Gray ; photograph


Type Yes to continue, anything else to finish:  
